In [ ]:
import os
import cv2
import lmdb
import pickle
import random
import numpy as np
from tqdm import tqdm
import torch

ROOT = r"D:\NUEVO\NUEVO"
LMDB_DIR = r"D:\NUEVO\Dataset_Preprocesado"   # Carpeta donde se guardarán los LMDB
os.makedirs(LMDB_DIR, exist_ok=True)

TRAIN_SUBJECTS = [1,10,11,12,14,2,3,4,6,7]
VAL_SUBJECTS   = [16,17,13]
TEST_SUBJECTS  = [8,9,15]

AUG_DIRS_TRAIN = ["rgb", "rgb_FLIP", "rgb_GRAY", "rgb_ROTATION", "rgb_STRETCH"]

CLASSES = ["A ver", "Aburrido", "Cansado", "Disgusto", "Feliz", "Huele mal",
           "Ladron","Llorar","Molesto","No","No sé","Si","Sorpresa","Triste"]

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ==================== UTILIDADES ====================
def imread_unicode(path):
    try:
        arr = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return img
    except Exception:
        return None

def collect_samples(root, subjects, for_train=False):
    samples = []
    for s in subjects:
        subj_name = f"SUJETO {s}"
        subj_path = os.path.join(root, subj_name, "señas_procesadas")
        if not os.path.isdir(subj_path):
            continue

        for class_name in os.listdir(subj_path):
            class_path = os.path.join(subj_path, class_name)
            if not os.path.isdir(class_path):
                continue

            for muestra in os.listdir(class_path):
                muestra_path = os.path.join(class_path, muestra)
                if not os.path.isdir(muestra_path):
                    continue

                if for_train:
                    for aug in AUG_DIRS_TRAIN:
                        rgb_dir = os.path.join(muestra_path, aug)
                        if os.path.isdir(rgb_dir) and any(f.lower().endswith(".png") for f in os.listdir(rgb_dir)):
                            samples.append((rgb_dir, class_name, subj_name, muestra, aug))
                else:
                    rgb_dir = os.path.join(muestra_path, "rgb")
                    if os.path.isdir(rgb_dir) and any(f.lower().endswith(".png") for f in os.listdir(rgb_dir)):
                        samples.append((rgb_dir, class_name, subj_name, muestra, "rgb"))
    return samples

# ==================== CONVERSIÓN A LMDB CON JPEG ====================
def save_lmdb(samples, out_path, jpeg_quality=90):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    MAP_SIZE = 200 * 1024**3   # 200 GB, amplio para tu dataset comprimido
    env = lmdb.open(out_path, map_size=MAP_SIZE)
    entry_idx = 0

    with env.begin(write=True) as txn:
        for rgb_dir, class_name, subj_name, muestra, augment in tqdm(samples, desc=f"Guardando {os.path.basename(out_path)}"):
            files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith(".png")])
            if len(files) == 0:
                continue

            frames_bytes = []
            for f in files:
                img = imread_unicode(os.path.join(rgb_dir, f))
                if img is None:
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                # Convertir a JPEG en memoria
                success, buf = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality])
                if not success:
                    continue
                frames_bytes.append(buf.tobytes())
            if len(frames_bytes) == 0:
                continue

            data = {
                'frames': frames_bytes,  # guardamos los frames como bytes JPEG
                'class': class_name,
                'sujeto': subj_name,
                'muestra': muestra,
                'augment': augment
            }
            txn.put(f"{entry_idx}".encode(), pickle.dumps(data))
            entry_idx += 1
    env.close()
    print(f"✓ LMDB guardado: {out_path} ({entry_idx} videos)")

# ==================== PASO DE PREPROCESAMIENTO ====================
print("Recolectando samples...")
train_samples = collect_samples(ROOT, TRAIN_SUBJECTS, for_train=True)
val_samples   = collect_samples(ROOT, VAL_SUBJECTS, for_train=False)
test_samples  = collect_samples(ROOT, TEST_SUBJECTS, for_train=False)

# Guardar en LMDB
save_lmdb(train_samples, os.path.join(LMDB_DIR, "train.lmdb"))
save_lmdb(val_samples, os.path.join(LMDB_DIR, "val.lmdb"))
save_lmdb(test_samples, os.path.join(LMDB_DIR, "test.lmdb"))


In [4]:
import os
import cv2
import lmdb
import pickle
import random
import numpy as np
from tqdm import tqdm
import torch

ROOT = r"D:\NUEVO\NUEVO"
LMDB_DIR = r"D:\NUEVO\Dataset_Preprocesado"   # Carpeta donde se guardarán los LMDB
os.makedirs(LMDB_DIR, exist_ok=True)

TRAIN_SUBJECTS = [1,10,11,12,14,2,3,4,6,7]
VAL_SUBJECTS   = [16,17,13]
TEST_SUBJECTS  = [8,9,15]

AUG_DIRS_TRAIN = ["rgb", "rgb_FLIP", "rgb_GRAY", "rgb_ROTATION", "rgb_STRETCH"]

CLASSES = ["A ver", "Aburrido", "Cansado", "Disgusto", "Feliz", "Huele mal",
           "Ladron","Llorar","Molesto","No","No sé","Si","Sorpresa","Triste"]

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ==================== UTILIDADES ====================
def imread_unicode(path):
    try:
        arr = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return img
    except Exception:
        return None

def collect_samples(root, subjects, for_train=False):
    samples = []
    for s in subjects:
        subj_name = f"SUJETO {s}"
        subj_path = os.path.join(root, subj_name, "señas_procesadas")
        if not os.path.isdir(subj_path):
            continue

        for class_name in os.listdir(subj_path):
            class_path = os.path.join(subj_path, class_name)
            if not os.path.isdir(class_path):
                continue

            for muestra in os.listdir(class_path):
                muestra_path = os.path.join(class_path, muestra)
                if not os.path.isdir(muestra_path):
                    continue

                if for_train:
                    for aug in AUG_DIRS_TRAIN:
                        rgb_dir = os.path.join(muestra_path, aug)
                        if os.path.isdir(rgb_dir) and any(f.lower().endswith(".png") for f in os.listdir(rgb_dir)):
                            samples.append((rgb_dir, class_name, subj_name, muestra, aug))
                else:
                    rgb_dir = os.path.join(muestra_path, "rgb")
                    if os.path.isdir(rgb_dir) and any(f.lower().endswith(".png") for f in os.listdir(rgb_dir)):
                        samples.append((rgb_dir, class_name, subj_name, muestra, "rgb"))
    return samples

In [5]:
# ==================== APPEND SOLO NUEVAS DEL SUJETO 1 A train.lmdb ====================
import os, pickle, lmdb, cv2
from tqdm import tqdm
import numpy as np

train_lmdb_path = os.path.join(LMDB_DIR, "train.lmdb")

def scan_lmdb_index_and_existing_tuples(lmdb_path):
    """
    Retorna:
      - next_idx: siguiente índice entero para nuevas claves
      - existing: set de tuplas (sujeto, clase, muestra, augment)
    """
    if not os.path.exists(lmdb_path):
        return 0, set()

    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    existing = set()
    max_idx = -1
    with env.begin() as txn:
        with txn.cursor() as cur:
            for k, v in cur:
                try:
                    idx = int(k.decode())
                    if idx > max_idx:
                        max_idx = idx
                except:
                    # si alguna clave no es numérica, ignórala para el next_idx
                    pass
                try:
                    data = pickle.loads(v)
                    key_tuple = (
                        data.get('sujeto', ''),
                        data.get('class', ''),
                        data.get('muestra', ''),
                        data.get('augment', '')
                    )
                    existing.add(key_tuple)
                except Exception:
                    continue
    env.close()
    return (max_idx + 1), existing

def jpeg_encode(img_bgr, q=90):
    ok, buf = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    if not ok:
        return None
    return buf.tobytes()

def append_samples_to_lmdb(lmdb_path, samples, jpeg_quality=90, map_gb=200):
    """
    Agrega samples (lista de tuplas: (rgb_dir, class_name, subj_name, muestra, augment))
    al LMDB existente, asignando claves continuas.
    """
    MAP_SIZE = map_gb * 1024**3
    env = lmdb.open(lmdb_path, map_size=MAP_SIZE)  # abre existente o crea
    next_idx, existing = scan_lmdb_index_and_existing_tuples(lmdb_path)

    # filtra solo nuevos por tupla
    to_add = []
    for (rgb_dir, class_name, subj_name, muestra, augment) in samples:
        key_tuple = (subj_name, class_name, muestra, augment)
        if key_tuple not in existing:
            to_add.append((rgb_dir, class_name, subj_name, muestra, augment))

    print(f"Encontradas {len(samples)} muestras del SUJETO 1; nuevas para agregar: {len(to_add)}")
    if not to_add:
        env.close()
        return

    with env.begin(write=True) as txn:
        for rgb_dir, class_name, subj_name, muestra, augment in tqdm(to_add, desc="Append a train.lmdb"):
            files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith(".png")])
            if not files:
                continue

            frames_bytes = []
            for f in files:
                img = imread_unicode(os.path.join(rgb_dir, f))  # tu helper
                if img is None:
                    continue
                # BGR->RGB? OJO: para guardar jpeg no importa el orden,
                # pero si quieres consistencia con tu pipeline previo (guardabas RGB),
                # deja la conversión:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                # Si prefieres guardar BGR (no recomendado si antes era RGB), cambia aquí.
                enc = cv2.imencode('.jpg', img_rgb, [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality])[1]
                frames_bytes.append(enc.tobytes())

            if not frames_bytes:
                continue

            data = {
                'frames': frames_bytes,
                'class': class_name,
                'sujeto': subj_name,
                'muestra': muestra,
                'augment': augment
            }
            txn.put(str(next_idx).encode(), pickle.dumps(data))
            next_idx += 1

    env.close()
    print("✓ Append completado.")

# 1) Recolectar SOLO SUJETO 1 con augs de train
train_s1 = collect_samples(ROOT, [1], for_train=True)   # [(rgb_dir, class, subj, muestra, augment), ...]

# 2) Agregar solo las nuevas al LMDB de train
append_samples_to_lmdb(train_lmdb_path, train_s1, jpeg_quality=90, map_gb=200)


Encontradas 1175 muestras del SUJETO 1; nuevas para agregar: 485


Append a train.lmdb: 100%|██████████| 485/485 [01:56<00:00,  4.15it/s]


✓ Append completado.


In [ ]:
import os
import lmdb
import pickle
import random
import cv2
import numpy as np
from tqdm import tqdm
from collections import defaultdict
import imageio

# ==================== CONFIGURACIÓN ====================
LMDB_PATH = r"D:\NUEVO\Dataset_Preprocesado\train.lmdb"
OUTPUT_DIR = r"D:\NUEVO\Dataset_Preprocesado"
SHOW_EXAMPLES = True   # True para generar GIFs de ejemplo
N_EXAMPLES = 3

# ==================== FUNCIÓN PARA DECODIFICAR FRAMES ====================
def decode_frames(frames_bytes):
    frames = []
    for b in frames_bytes:
        arr = np.frombuffer(b, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is not None:
            frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    return frames

# ==================== FUNCIÓN PRINCIPAL DE VERIFICACIÓN ====================
def verificar_lmdb(lmdb_path):
    print(f"\n🔍 Verificando dataset LMDB en: {lmdb_path}\n")

    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    total = 0
    errores = {}
    dist_clases = defaultdict(int)

    with env.begin() as txn:
        cursor = txn.cursor()
        for key, value in tqdm(cursor, desc="Verificando entradas LMDB"):
            try:
                entry = pickle.loads(value)

                # Validar estructura
                if not isinstance(entry, dict):
                    raise TypeError("Entrada no es un diccionario")

                # Leer campos esperados
                class_name = entry.get("class", "Desconocida")
                sujeto = entry.get("sujeto", "N/A")
                muestra = entry.get("muestra", "N/A")
                augment = entry.get("augment", "N/A")
                frames = entry.get("frames", [])

                if not isinstance(frames, list) or len(frames) == 0:
                    raise ValueError("Sin frames válidos")

                dist_clases[class_name] += 1
                total += 1

            except Exception as e:
                errores[int(key.decode())] = str(e)
                dist_clases["Desconocida"] += 1

    env.close()

    # ==================== RESUMEN ====================
    print("\n📊 Resumen general:")
    print(f" - Total de muestras: {total + len(errores)}")
    print(f" - Entradas correctas: {total}")
    print(f" - Entradas con errores: {len(errores)}\n")

    print("📂 Distribución por clase:")
    for c, n in dist_clases.items():
        print(f"   {c:<20} ->  {n:4d}")

    if errores:
        print("\n⚠️  Detalles de errores (máx 10):")
        for i, (k, v) in enumerate(list(errores.items())[:10]):
            print(f"  {i+1:02d}. {k} → {v}")

    return total, errores, dist_clases

# ==================== MOSTRAR EJEMPLOS ====================
def mostrar_ejemplos(lmdb_path, output_dir, n=3):
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    keys = []

    with env.begin() as txn:
        cursor = txn.cursor()
        for key, _ in cursor:
            keys.append(int(key.decode()))

    random.shuffle(keys)
    keys = keys[:n]

    for k in keys:
        with env.begin() as txn:
            entry = pickle.loads(txn.get(str(k).encode()))
            clase = entry["class"]
            sujeto = entry["sujeto"]
            aumento = entry["augment"]
            frames = decode_frames(entry["frames"])

        print(f"\n🎥 Ejemplo {k}: {clase} ({sujeto} - {aumento})")
        print(f"Frames decodificados: {len(frames)}")

        if len(frames) > 0:
            gif_path = os.path.join(output_dir, f"preview_{k}.gif")
            imageio.mimsave(gif_path, frames, fps=10)
            print(f"💾 GIF guardado en: {gif_path}")

    env.close()

# ==================== EJECUCIÓN ====================
total, errores, dist = verificar_lmdb(LMDB_PATH)

if SHOW_EXAMPLES:
    mostrar_ejemplos(LMDB_PATH, OUTPUT_DIR, N_EXAMPLES)

print("\n✅ Verificación completada.")